# 02 Data Cleaning

This notebook cleans the raw Bike Share Toronto trip data and Toronto hourly weather data.
The goal is to create analysis-ready datasets for SQL, dashboarding, and station-level imbalance analysis.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

project_root = Path("..")

bike_path = project_root / "data" / "raw" / "bikeshare-ridership-2026.csv"
weather_folder = project_root / "data" / "raw" / "weather"

weather_files = [
    weather_folder / "en_climate_hourly_ON_6158355_01-2026_P1H.csv",
    weather_folder / "en_climate_hourly_ON_6158355_02-2026_P1H.csv",
    weather_folder / "en_climate_hourly_ON_6158355_03-2026_P1H.csv",
]

trips = pd.read_csv(bike_path)

weather = pd.concat(
    [pd.read_csv(file) for file in weather_files],
    ignore_index=True
)

print("Trips:", trips.shape)
print("Weather:", weather.shape)

Trips: (552073, 11)
Weather: (2160, 31)


In [2]:
def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
        .str.replace("°", "", regex=False)
        .str.replace(".", "", regex=False)
    )
    return df

trips_clean = clean_column_names(trips)
weather_clean = clean_column_names(weather)

print(trips_clean.columns.tolist())
print(weather_clean.columns.tolist())

['trip_id', 'trip_duration', 'start_station_id', 'start_time', 'start_station_name', 'end_station_id', 'end_time', 'end_station_name', 'bike_id', 'user_type', 'bike_model']
['longitude_x', 'latitude_y', 'station_name', 'climate_id', 'date_time_lst', 'year', 'month', 'day', 'time_lst', 'flag', 'temp_c', 'temp_flag', 'dew_point_temp_c', 'dew_point_temp_flag', 'rel_hum_%', 'rel_hum_flag', 'precip_amount_mm', 'precip_amount_flag', 'wind_dir_10s_deg', 'wind_dir_flag', 'wind_spd_km_h', 'wind_spd_flag', 'visibility_km', 'visibility_flag', 'stn_press_kpa', 'stn_press_flag', 'hmdx', 'hmdx_flag', 'wind_chill', 'wind_chill_flag', 'weather']


In [3]:
weather_clean = weather_clean.rename(columns={
    "longitude_x": "longitude",
    "latitude_y": "latitude",
    "station_name": "weather_station_name",
    "climate_id": "climate_id",
    "date_time_lst": "weather_datetime",
    "temp_c": "temperature_c",
    "dew_point_temp_c": "dew_point_c",
    "rel_hum_%": "relative_humidity",
    "precip_amount_mm": "precipitation_mm",
    "stn_press_kpa": "station_pressure_kpa"
})

print(weather_clean.columns.tolist())

['longitude', 'latitude', 'weather_station_name', 'climate_id', 'weather_datetime', 'year', 'month', 'day', 'time_lst', 'flag', 'temperature_c', 'temp_flag', 'dew_point_c', 'dew_point_temp_flag', 'relative_humidity', 'rel_hum_flag', 'precipitation_mm', 'precip_amount_flag', 'wind_dir_10s_deg', 'wind_dir_flag', 'wind_spd_km_h', 'wind_spd_flag', 'visibility_km', 'visibility_flag', 'station_pressure_kpa', 'stn_press_flag', 'hmdx', 'hmdx_flag', 'wind_chill', 'wind_chill_flag', 'weather']


In [4]:
trips_clean["start_time"] = pd.to_datetime(trips_clean["start_time"], errors="coerce")
trips_clean["end_time"] = pd.to_datetime(trips_clean["end_time"], errors="coerce")

weather_clean["weather_datetime"] = pd.to_datetime(weather_clean["weather_datetime"], errors="coerce")

print("Invalid start times:", trips_clean["start_time"].isna().sum())
print("Invalid end times:", trips_clean["end_time"].isna().sum())
print("Invalid weather datetimes:", weather_clean["weather_datetime"].isna().sum())

Invalid start times: 0
Invalid end times: 498
Invalid weather datetimes: 0


In [5]:
trips_clean["start_date"] = trips_clean["start_time"].dt.date
trips_clean["start_hour"] = trips_clean["start_time"].dt.hour
trips_clean["start_month"] = trips_clean["start_time"].dt.month
trips_clean["start_month_name"] = trips_clean["start_time"].dt.month_name()
trips_clean["start_weekday"] = trips_clean["start_time"].dt.day_name()
trips_clean["start_weekday_num"] = trips_clean["start_time"].dt.weekday
trips_clean["is_weekend"] = trips_clean["start_weekday_num"].isin([5, 6])

trips_clean["start_time_hour"] = trips_clean["start_time"].dt.floor("h")
trips_clean["end_time_hour"] = trips_clean["end_time"].dt.floor("h")

trips_clean[[
    "start_time",
    "start_date",
    "start_hour",
    "start_month_name",
    "start_weekday",
    "is_weekend",
    "start_time_hour"
]].head()

,start_time,start_date,start_hour,start_month_name,start_weekday,is_weekend,start_time_hour
0,2026-01-01 00:00:26,2026-01-01,0,January,Thursday,False,2026-01-01
1,2026-01-01 00:02:05,2026-01-01,0,January,Thursday,False,2026-01-01
2,2026-01-01 00:04:15,2026-01-01,0,January,Thursday,False,2026-01-01
3,2026-01-01 00:04:29,2026-01-01,0,January,Thursday,False,2026-01-01
4,2026-01-01 00:04:53,2026-01-01,0,January,Thursday,False,2026-01-01


In [6]:
trips_clean["duration_minutes"] = trips_clean["trip_duration"] / 60

trips_clean["valid_duration"] = (
    (trips_clean["duration_minutes"] > 0) &
    (trips_clean["duration_minutes"] <= 180)
)

print("Invalid duration rows:", (~trips_clean["valid_duration"]).sum())
print("Valid duration rows:", trips_clean["valid_duration"].sum())

trips_clean["duration_minutes"].describe()

Invalid duration rows: 1840
Valid duration rows: 550233


count    552073.000000
mean         12.236470
std          16.261877
min           0.000000
25%           6.200000
50%           9.683333
75%          14.983333
max        3455.566667
Name: duration_minutes, dtype: float64

In [8]:
trips_clean["is_completed_trip"] = (
    trips_clean["end_time"].notna() &
    trips_clean["end_station_id"].notna() &
    trips_clean["end_station_name"].notna()
)

print(trips_clean["is_completed_trip"].value_counts())
print("Incomplete trip percentage:", round((~trips_clean["is_completed_trip"]).mean() * 100, 2), "%") 

is_completed_trip
True     550398
False      1675
Name: count, dtype: int64
Incomplete trip percentage: 0.3 %


In [9]:
weather_keep_cols = [
    "weather_datetime",
    "year",
    "month",
    "day",
    "time_lst",
    "temperature_c",
    "dew_point_c",
    "relative_humidity",
    "precipitation_mm",
    "station_pressure_kpa",
    "weather_station_name",
    "climate_id",
    "longitude",
    "latitude"
]

weather_clean = weather_clean[weather_keep_cols].copy()

weather_clean["weather_date"] = weather_clean["weather_datetime"].dt.date
weather_clean["weather_hour"] = weather_clean["weather_datetime"].dt.hour

weather_clean["has_precipitation"] = weather_clean["precipitation_mm"].fillna(0) > 0

weather_clean.head()

,weather_datetime,year,month,day,time_lst,temperature_c,dew_point_c,relative_humidity,precipitation_mm,station_pressure_kpa,weather_station_name,climate_id,longitude,latitude,weather_date,weather_hour,has_precipitation
0,2026-01-01 00:00:00,2026,1,1,00:00,-9.1,-13.9,68.0,0.0,98.88,TORONTO CITY,6158355,-79.4,43.67,2026-01-01,0,False
1,2026-01-01 01:00:00,2026,1,1,01:00,-10.3,-14.7,70.0,0.0,99.00,TORONTO CITY,6158355,-79.4,43.67,2026-01-01,1,False
2,2026-01-01 02:00:00,2026,1,1,02:00,-10.7,-15.0,70.0,0.0,99.10,TORONTO CITY,6158355,-79.4,43.67,2026-01-01,2,False
3,2026-01-01 03:00:00,2026,1,1,03:00,-10.6,-16.3,63.0,0.0,99.22,TORONTO CITY,6158355,-79.4,43.67,2026-01-01,3,False
4,2026-01-01 04:00:00,2026,1,1,04:00,-10.5,-18.7,51.0,0.0,99.27,TORONTO CITY,6158355,-79.4,43.67,2026-01-01,4,False


In [10]:
weather_clean.isna().sum()

weather_datetime        0
year                    0
month                   0
day                     0
time_lst                0
temperature_c           5
dew_point_c             5
relative_humidity       5
precipitation_mm        5
station_pressure_kpa    5
weather_station_name    0
climate_id              0
longitude               0
latitude                0
weather_date            0
weather_hour            0
has_precipitation       0
dtype: int64

In [11]:
trips_weather = trips_clean.merge(
    weather_clean,
    left_on="start_time_hour",
    right_on="weather_datetime",
    how="left"
)

print("Trips before weather join:", trips_clean.shape)
print("Trips after weather join:", trips_weather.shape)
print("Trips missing weather:", trips_weather["temperature_c"].isna().sum())

Trips before weather join: (552073, 23)
Trips after weather join: (552073, 40)
Trips missing weather: 498


In [12]:
station_hourly_departures = (
    trips_clean
    .groupby([
        "start_station_id",
        "start_station_name",
        "start_time_hour"
    ])
    .size()
    .reset_index(name="departures")
)

station_hourly_departures = station_hourly_departures.rename(columns={
    "start_station_id": "station_id",
    "start_station_name": "station_name",
    "start_time_hour": "datetime_hour"
})

station_hourly_departures.head()

,station_id,station_name,datetime_hour,departures
0,7000,Fort York Blvd / Capreol Ct,2026-01-01 01:00:00,1
1,7000,Fort York Blvd / Capreol Ct,2026-01-01 10:00:00,2
2,7000,Fort York Blvd / Capreol Ct,2026-01-02 08:00:00,2
3,7000,Fort York Blvd / Capreol Ct,2026-01-02 09:00:00,2
4,7000,Fort York Blvd / Capreol Ct,2026-01-02 10:00:00,1


In [13]:
completed_trips = trips_clean[trips_clean["is_completed_trip"]].copy()

station_hourly_arrivals = (
    completed_trips
    .groupby([
        "end_station_id",
        "end_station_name",
        "end_time_hour"
    ])
    .size()
    .reset_index(name="arrivals")
)

station_hourly_arrivals = station_hourly_arrivals.rename(columns={
    "end_station_id": "station_id",
    "end_station_name": "station_name",
    "end_time_hour": "datetime_hour"
})

station_hourly_arrivals.head()

,station_id,station_name,datetime_hour,arrivals
0,7000.0,Fort York Blvd / Capreol Ct,2026-01-01 12:00:00,1
1,7000.0,Fort York Blvd / Capreol Ct,2026-01-01 13:00:00,1
2,7000.0,Fort York Blvd / Capreol Ct,2026-01-01 19:00:00,1
3,7000.0,Fort York Blvd / Capreol Ct,2026-01-01 20:00:00,1
4,7000.0,Fort York Blvd / Capreol Ct,2026-01-01 21:00:00,1


In [14]:
station_hourly_flow = station_hourly_departures.merge(
    station_hourly_arrivals,
    on=["station_id", "datetime_hour"],
    how="outer",
    suffixes=("_departure", "_arrival")
)

station_hourly_flow["station_name"] = station_hourly_flow["station_name_departure"].fillna(
    station_hourly_flow["station_name_arrival"]
)

station_hourly_flow["departures"] = station_hourly_flow["departures"].fillna(0).astype(int)
station_hourly_flow["arrivals"] = station_hourly_flow["arrivals"].fillna(0).astype(int)

station_hourly_flow["net_flow"] = station_hourly_flow["arrivals"] - station_hourly_flow["departures"]

station_hourly_flow = station_hourly_flow[
    ["station_id", "station_name", "datetime_hour", "departures", "arrivals", "net_flow"]
].copy()

station_hourly_flow["date"] = station_hourly_flow["datetime_hour"].dt.date
station_hourly_flow["hour"] = station_hourly_flow["datetime_hour"].dt.hour
station_hourly_flow["weekday"] = station_hourly_flow["datetime_hour"].dt.day_name()
station_hourly_flow["is_weekend"] = station_hourly_flow["datetime_hour"].dt.weekday.isin([5, 6])

station_hourly_flow.head()

,station_id,station_name,datetime_hour,departures,arrivals,net_flow,date,hour,weekday,is_weekend
0,7000.0,Fort York Blvd / Capreol Ct,2026-01-01 01:00:00,1,0,-1,2026-01-01,1,Thursday,False
1,7000.0,Fort York Blvd / Capreol Ct,2026-01-01 10:00:00,2,0,-2,2026-01-01,10,Thursday,False
2,7000.0,Fort York Blvd / Capreol Ct,2026-01-01 12:00:00,0,1,1,2026-01-01,12,Thursday,False
3,7000.0,Fort York Blvd / Capreol Ct,2026-01-01 13:00:00,0,1,1,2026-01-01,13,Thursday,False
4,7000.0,Fort York Blvd / Capreol Ct,2026-01-01 19:00:00,0,1,1,2026-01-01,19,Thursday,False


In [15]:
station_hourly_flow.sort_values("net_flow").head(20)

,station_id,station_name,datetime_hour,departures,arrivals,net_flow,date,hour,weekday,is_weekend
18199,7019.0,Temperance St Station,2026-03-09 17:00:00,59,3,-56,2026-03-09,17,Monday,False
18513,7019.0,Temperance St Station,2026-03-30 17:00:00,55,6,-49,2026-03-30,17,Monday,False
13791,7015.0,King St W / Bay St (West Side),2026-03-31 17:00:00,40,3,-37,2026-03-31,17,Tuesday,False
18217,7019.0,Temperance St Station,2026-03-10 17:00:00,38,1,-37,2026-03-10,17,Tuesday,False
13518,7015.0,King St W / Bay St (West Side),2026-03-10 17:00:00,38,3,-35,2026-03-10,17,Tuesday,False
18530,7019.0,Temperance St Station,2026-03-31 17:00:00,34,0,-34,2026-03-31,17,Tuesday,False
13500,7015.0,King St W / Bay St (West Side),2026-03-09 17:00:00,35,3,-32,2026-03-09,17,Monday,False
13774,7015.0,King St W / Bay St (West Side),2026-03-30 17:00:00,40,9,-31,2026-03-30,17,Monday,False
18126,7019.0,Temperance St Station,2026-03-04 17:00:00,32,3,-29,2026-03-04,17,Wednesday,False
18218,7019.0,Temperance St Station,2026-03-10 18:00:00,31,2,-29,2026-03-10,18,Tuesday,False


In [16]:
station_hourly_flow.sort_values("net_flow", ascending=False).head(20)

,station_id,station_name,datetime_hour,departures,arrivals,net_flow,date,hour,weekday,is_weekend
13765,7015.0,King St W / Bay St (West Side),2026-03-30 08:00:00,3,139,136,2026-03-30,8,Monday,False
13766,7015.0,King St W / Bay St (West Side),2026-03-30 09:00:00,0,96,96,2026-03-30,9,Monday,False
18504,7019.0,Temperance St Station,2026-03-30 08:00:00,0,90,90,2026-03-30,8,Monday,False
13509,7015.0,King St W / Bay St (West Side),2026-03-10 08:00:00,0,85,85,2026-03-10,8,Tuesday,False
12887,7015.0,King St W / Bay St (West Side),2026-01-13 08:00:00,0,84,84,2026-01-13,8,Tuesday,False
13711,7015.0,King St W / Bay St (West Side),2026-03-26 09:00:00,2,83,81,2026-03-26,9,Thursday,False
18190,7019.0,Temperance St Station,2026-03-09 08:00:00,0,81,81,2026-03-09,8,Monday,False
13710,7015.0,King St W / Bay St (West Side),2026-03-26 08:00:00,1,79,78,2026-03-26,8,Thursday,False
13491,7015.0,King St W / Bay St (West Side),2026-03-09 08:00:00,1,78,77,2026-03-09,8,Monday,False
13492,7015.0,King St W / Bay St (West Side),2026-03-09 09:00:00,0,71,71,2026-03-09,9,Monday,False


In [18]:
processed_folder = project_root / "data" / "processed"
processed_folder.mkdir(parents=True, exist_ok=True)

trips_clean.to_csv(processed_folder / "clean_trips.csv", index=False)
weather_clean.to_csv(processed_folder / "clean_weather_hourly.csv", index=False)
trips_weather.to_csv(processed_folder / "trips_with_weather.csv", index=False)
station_hourly_flow.to_csv(processed_folder / "station_hourly_flow.csv", index=False)

print("Cleaned files exported.")

Cleaned files exported.
